# Evaluation of resource-allocation policies

This notebook is the reproducible source for the optimization evaluation and the staffing management question. It replaces the previous results, which silently used the legacy start-to-complete duration target. Every simulation below explicitly uses the active lifecycle introduced on `refac/event_lifecycle`: active service sessions are sampled from `simulation_inputs_active.json`, while suspensions, resumptions, aborts, withdrawals, and non-active waiting remain separate lifecycle mechanisms.

The experiment uses the deployed BPMN/Petri-net process, visit-aware branching, contextual OrgModel permissions, calendar availability, and common random numbers (CRN). The fitted distribution sampler is used because the processing-time analysis found it to be the most reliable of the three active-session samplers. All choices are recorded alongside the results.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = next(
    path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "analysis").is_dir() and (path / "simulation").is_dir()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from analysis.availability import YearlyAvailability
from analysis.tum_style import (
    TUM_BLUE, TUM_GRAY, TUM_GRAY_DARK, TUM_ORANGE, TUM_RED, TUM_TEAL,
    apply_tum_style, save_figure,
)
from scripts import opt_metrics as om
from scripts import run_experiments as R

apply_tum_style()
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

DAY = 86_400.0
RESULT_DIR = ROOT / "output" / "evaluation"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Experimental design and provenance

Four paired replications balance precision and notebook runtime. A 10-day horizon permits a substantial number of cases to finish while keeping the complete 20-run study practical on a laptop. Confidence intervals use Student's *t* distribution and should be read as simulation uncertainty, not uncertainty about the historical population.

The policies are R-RMA (`random`), Piled Execution (`piled`), and the middle k-Batching setting (`k=5`) as a representative batching policy. Repeating the entire k-sweep here would add no new evaluation concept and is disproportionately expensive under the active lifecycle; the dedicated k-Batching subsection covers parameter sensitivity. Cycle time is calculated only for naturally completed cases; completion counts and milestone reach rates are reported with it to expose horizon truncation and survivorship effects. Resource metrics include every human in the active OrgModel pool, including staff who perform no work in a run. The calendar-designated automated account remains part of the process but is excluded from staffing metrics.

In [ ]:
SEEDS = [1, 2, 3, 4]
HORIZON_DAYS = 10
POLICIES = ["random", "piled", "kbatch5"]
POLICY_LABELS = {
    "random": "R-RMA",
    "piled": "R-PE",
    "kbatch5": "k=5",
}

RUN_CONFIG = {
    "horizon_days": HORIZON_DAYS,
    "seeds": SEEDS,
    "scenario": "normal",
    "crn": True,
    "process_model": "advanced",
    "branching_mode": "visit",
    "permissions": "orgmodel",
    "lifecycle_mode": "active",
    "processing_time_mode": "distribution",
}

calendar = YearlyAvailability.from_json(ROOT / "models" / "availability_model.json")
permissions, _ = R.load_permission_model(RUN_CONFIG["permissions"], seed=SEEDS[0])
resource_pool = set(permissions.resources())
human_resources = resource_pool - calendar.system
automated_resources = resource_pool & calendar.system

print(json.dumps(RUN_CONFIG, indent=2))
print(
    f"Active pool: {len(resource_pool)} resources = {len(human_resources)} human "
    f"+ {len(automated_resources)} automated {sorted(automated_resources)}"
)

In [ ]:
def ci95(values):
    values = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(values) < 2:
        return np.nan
    return float(stats.t.ppf(0.975, len(values) - 1) * values.std(ddof=1) / np.sqrt(len(values)))


def run_and_measure(policy, seed, *, excluded=None, resource_subset=None, cache_key="policy"):
    subset = human_resources if resource_subset is None else set(resource_subset)
    excluded_set = set(excluded or ())
    cache_dir = RESULT_DIR / "run_cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f"{cache_key}_{policy}_seed{seed}.json"
    cache_configuration = {
        **RUN_CONFIG,
        "policy": policy,
        "seed": seed,
        "excluded_resources": sorted(excluded_set),
        "resource_subset": sorted(subset),
    }
    if cache_path.exists():
        cached = json.loads(cache_path.read_text())
        if cached.get("cache_configuration") == cache_configuration:
            print(f"cached   policy={policy:>8} seed={seed} completed={cached['row']['completed_cases']:4d}")
            return cached

    df, meta = R.run_once(
        policy,
        seed,
        HORIZON_DAYS,
        RUN_CONFIG["scenario"],
        RUN_CONFIG["crn"],
        RUN_CONFIG["process_model"],
        RUN_CONFIG["branching_mode"],
        lifecycle_mode=RUN_CONFIG["lifecycle_mode"],
        processing_time_mode=RUN_CONFIG["processing_time_mode"],
        permissions=RUN_CONFIG["permissions"],
        excluded_override=excluded,
    )
    expected = {
        "horizon_days": HORIZON_DAYS,
        "scenario": RUN_CONFIG["scenario"],
        "crn": RUN_CONFIG["crn"],
        "process_model": RUN_CONFIG["process_model"],
        "branching_mode": RUN_CONFIG["branching_mode"],
        "permissions": RUN_CONFIG["permissions"],
        "lifecycle_mode": RUN_CONFIG["lifecycle_mode"],
        "processing_time_mode": RUN_CONFIG["processing_time_mode"],
    }
    for key, value in expected.items():
        assert meta["configuration"][key] == value, (key, meta["configuration"])
    assert "work_item_id" in df.columns and df["work_item_id"].notna().any()

    metrics = om.evaluate(
        df,
        arrival_times=meta["arrival_times"],
        availability_seconds=meta["availability_seconds"],
        completed_case_ids=meta["completed_case_ids"],
        resource_subset=subset,
    )
    custom = metrics["custom_metrics"]
    arrivals = len(meta["arrival_times"])
    completed = metrics["case_filter"]["n_cases_completed"]
    first_offer = custom["time_to_first_offer"]
    decision = custom["time_to_decision"]
    row = {
        "policy": policy,
        "seed": seed,
        "excluded_resources": ",".join(meta["configuration"]["excluded_resources"]),
        "arrivals": arrivals,
        "completed_cases": completed,
        "completion_share": completed / arrivals if arrivals else np.nan,
        "cycle_time_days": metrics["cycle_time"]["avg_cycle_time_s"] / DAY,
        "p95_cycle_days": metrics["cycle_time"]["p95_cycle_time_s"] / DAY,
        "occupation": metrics["occupation"]["avg_resource_occupation"],
        "fairness": metrics["fairness"]["resource_fairness"],
        "time_to_first_offer_days": first_offer["mean_s"] / DAY,
        "first_offer_reach_share": first_offer["n_cases_reaching_it"] / arrivals if arrivals else np.nan,
        "time_to_decision_days": decision["mean_s"] / DAY,
        "decision_reach_share": decision["n_cases_reaching_it"] / arrivals if arrivals else np.nan,
        "case_handover_rate": custom["handover_rate"]["handover_rate"],
        "activity_switch_rate": custom["resource_activity_switch_rate"]["activity_switch_rate"],
        "rolling_workload_std": custom["rolling_workload_balance"]["mean_window_std"],
        "resources_evaluated": metrics["occupation"]["n_resources_evaluated"],
        "zero_availability_resources": metrics["occupation"]["n_resources_zero_availability"],
    }
    instances = om.paired_instances(df)
    human_counts = instances.loc[
        instances["resource"].isin(human_resources), "resource"
    ].value_counts()
    result = {
        "cache_configuration": cache_configuration,
        "row": row,
        "resource_occupation": metrics["occupation"]["per_resource"],
        "human_throughput_share": (
            (human_counts / human_counts.sum()).to_dict() if human_counts.sum() else {}
        ),
    }
    cache_path.write_text(json.dumps(result, indent=2, sort_keys=True) + "\n")
    print(f"finished policy={policy:>8} seed={seed} completed={completed:4d}")
    return result


def summarize(rows, group="policy"):
    frame = pd.DataFrame(rows)
    metrics = [
        "cycle_time_days", "p95_cycle_days", "occupation", "fairness",
        "completed_cases", "completion_share", "time_to_first_offer_days",
        "first_offer_reach_share", "time_to_decision_days", "decision_reach_share",
        "case_handover_rate", "activity_switch_rate", "rolling_workload_std",
    ]
    records = []
    for name, part in frame.groupby(group, sort=False):
        record = {group: name, "n_replications": len(part)}
        for metric in metrics:
            record[metric] = part[metric].mean()
            record[f"{metric}_ci95"] = ci95(part[metric])
        records.append(record)
    return pd.DataFrame(records).set_index(group), frame

## 2. Policy comparison

The following cell performs 12 active-lifecycle simulations. Each run is checkpointed as JSON immediately after completion. Rerunning the notebook reuses a checkpoint only when its full configuration matches, so an interrupted k-Batching run does not force the completed replications to be repeated.

In [ ]:
policy_runs = {
    policy: [run_and_measure(policy, seed) for seed in SEEDS]
    for policy in POLICIES
}
policy_summary, policy_run_metrics = summarize(
    [run["row"] for runs in policy_runs.values() for run in runs]
)

policy_run_metrics.to_csv(RESULT_DIR / "policy_run_metrics.csv", index=False)
policy_summary.to_csv(RESULT_DIR / "policy_summary.csv")
(RESULT_DIR / "configuration.json").write_text(json.dumps({
    **RUN_CONFIG,
    "policies": POLICIES,
    "resource_pool_size": len(resource_pool),
    "human_resource_count": len(human_resources),
    "automated_resources": sorted(automated_resources),
}, indent=2) + "\n")

display_columns = [
    "cycle_time_days", "completed_cases", "completion_share", "occupation",
    "fairness", "activity_switch_rate", "time_to_decision_days", "decision_reach_share",
]
policy_summary[display_columns].rename(index=POLICY_LABELS).round(3)

In [ ]:
def paired_deltas(frame, baseline="random"):
    columns = ["cycle_time_days", "completed_cases", "activity_switch_rate", "time_to_decision_days"]
    records = []
    base = frame[frame["policy"] == baseline].set_index("seed")
    for policy in POLICIES:
        if policy == baseline:
            continue
        other = frame[frame["policy"] == policy].set_index("seed")
        common = base.index.intersection(other.index)
        record = {"policy_vs_random": policy, "n_pairs": len(common)}
        for column in columns:
            delta = other.loc[common, column] - base.loc[common, column]
            record[f"delta_{column}"] = delta.mean()
            record[f"delta_{column}_ci95"] = ci95(delta)
        records.append(record)
    return pd.DataFrame(records).set_index("policy_vs_random")


policy_deltas = paired_deltas(policy_run_metrics)
policy_deltas.to_csv(RESULT_DIR / "policy_paired_deltas.csv")
policy_deltas.round(3)

The compact trade-off plot is retained because it combines the two outcome measures needed to detect survivorship: lower cycle time is only desirable when it is not obtained by completing fewer cases. The remaining measures are more legible in the tables than as separate charts.

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 3.7))
colors = [TUM_BLUE, TUM_ORANGE, TUM_TEAL]
for color, policy in zip(colors, POLICIES):
    row = policy_summary.loc[policy]
    ax.errorbar(
        row["cycle_time_days"], row["completed_cases"],
        xerr=row["cycle_time_days_ci95"], yerr=row["completed_cases_ci95"],
        fmt="o", ms=6, capsize=3, color=color, label=POLICY_LABELS[policy],
    )
ax.set(
    xlabel="Mean cycle time of completed cases (days; lower is better)",
    ylabel=f"Completed cases in {HORIZON_DAYS} days (higher is better)",
    title="Policy trade-off under the active lifecycle",
)
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
save_figure(fig, "04_policy_tradeoff")
plt.show()

### Operational diagnostics

The case-handover rate describes continuity within one customer case. It is not a direct test of Piled Execution. Piled Execution instead tries to keep a resource on the same activity, so `activity_switch_rate` is the relevant mechanism measure. Milestone means are accompanied by the share of arrivals that reach each milestone; comparing means alone would condition on a policy-dependent subset of cases.

In [ ]:
diagnostic_columns = [
    "time_to_first_offer_days", "first_offer_reach_share",
    "time_to_decision_days", "decision_reach_share",
    "case_handover_rate", "activity_switch_rate", "rolling_workload_std",
]
policy_summary[diagnostic_columns].rename(index=POLICY_LABELS).round(3)

## 3. Staffing question: which two employees can be removed with the least operational harm?

The previous report treated a 17-resource legacy map as the deployed workforce and claimed that `User_5` was an exclusive permission holder. The current OrgModel contains 144 resources. We therefore recompute the candidate pool and permission redundancy from the model actually used by the simulator.

Criticality is a screening score, not a causal estimate. It combines average occupation, throughput share, and permission breadth. Any human who is the sole candidate for a recognized `(activity, case type, weekday)` context receives a hard penalty. The leave-two-out simulations then test the two lowest-scoring employees against a deliberately high-criticality contrast under the same paired seeds.

In [ ]:
# Contextual permission redundancy in the deployed OrgModel.
model_case_types = sorted({
    case_type
    for entries in permissions._index.values()
    for _, case_type, _ in entries
    if case_type
}) or [None]
model_weekdays = pd.date_range("2016-01-04", periods=7, freq="D")
sole_candidate_contexts = pd.Series(0, index=sorted(human_resources), dtype=int)
recognized_contexts = 0
for activity in permissions._index:
    for case_type in model_case_types:
        for when in model_weekdays:
            candidates = set(permissions.candidates(activity, case_type=case_type, when=when)) & human_resources
            if candidates:
                recognized_contexts += 1
            if len(candidates) == 1:
                sole_candidate_contexts.loc[next(iter(candidates))] += 1

base_runs = policy_runs["random"]
occupation_by_resource = pd.DataFrame([
    run["resource_occupation"] for run in base_runs
]).reindex(columns=sorted(human_resources), fill_value=0.0).mean()

throughput_frames = [
    pd.Series(run["human_throughput_share"], dtype=float) for run in base_runs
]
throughput_share = pd.concat(throughput_frames, axis=1).reindex(sorted(human_resources)).fillna(0.0).mean(axis=1)

permission_breadth = pd.Series({
    resource: sum(
        any(resource in members for members, _, _ in entries)
        for entries in permissions._index.values()
    )
    for resource in human_resources
})

def normalize(series):
    span = series.max() - series.min()
    return (series - series.min()) / span if span else pd.Series(0.0, index=series.index)

criticality = pd.DataFrame({
    "occupation": occupation_by_resource,
    "throughput_share": throughput_share,
    "permission_breadth": permission_breadth,
    "sole_candidate_contexts": sole_candidate_contexts,
}).reindex(sorted(human_resources)).fillna(0.0)
criticality["criticality_score"] = (
    0.4 * normalize(criticality["occupation"])
    + 0.4 * normalize(criticality["throughput_share"])
    + 0.2 * normalize(criticality["permission_breadth"])
    + (criticality["sole_candidate_contexts"] > 0).astype(float)
)
criticality = criticality.sort_values("criticality_score")
criticality.to_csv(RESULT_DIR / "resource_criticality.csv")

print(f"Recognized OrgModel contexts checked: {recognized_contexts}")
print(f"Single-human-candidate contexts: {int(sole_candidate_contexts.sum())}")
display(pd.concat([criticality.head(6), criticality.tail(6)]).round(4))

In [ ]:
remove_low = list(criticality.index[:2])
remove_high = list(criticality.index[-2:])
print("Low-criticality pair:", remove_low)
print("High-criticality contrast:", remove_high)

staffing_runs = {
    "baseline": base_runs,
    "remove_low": [
        run_and_measure(
            "random", seed, excluded=set(remove_low),
            resource_subset=human_resources - set(remove_low), cache_key="remove_low",
        )
        for seed in SEEDS
    ],
    "remove_high": [
        run_and_measure(
            "random", seed, excluded=set(remove_high),
            resource_subset=human_resources - set(remove_high), cache_key="remove_high",
        )
        for seed in SEEDS
    ],
}

staffing_summary, staffing_run_metrics = summarize([
    {**run["row"], "staffing_scenario": scenario}
    for scenario, runs in staffing_runs.items()
    for run in runs
], group="staffing_scenario")
staffing_run_metrics.to_csv(RESULT_DIR / "staffing_run_metrics.csv", index=False)
staffing_summary.to_csv(RESULT_DIR / "staffing_summary.csv")
staffing_summary[[
    "cycle_time_days", "completed_cases", "completion_share", "occupation",
    "fairness", "time_to_decision_days", "decision_reach_share",
]].round(3)

In [ ]:
baseline = staffing_summary.loc["baseline"]
impact = pd.DataFrame({
    scenario: {
        "Cycle time": 100 * (staffing_summary.loc[scenario, "cycle_time_days"] / baseline["cycle_time_days"] - 1),
        "Completed cases": 100 * (staffing_summary.loc[scenario, "completed_cases"] / baseline["completed_cases"] - 1),
        "Time to decision": 100 * (staffing_summary.loc[scenario, "time_to_decision_days"] / baseline["time_to_decision_days"] - 1),
    }
    for scenario in ["remove_low", "remove_high"]
}).T
impact.to_csv(RESULT_DIR / "staffing_relative_impact.csv")
impact.round(2)

The staffing chart is retained because the signs of the changes matter: a lower cycle-time mean can coexist with fewer completions and therefore indicate survivorship rather than improvement. The table remains the source for exact values.

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 3.6))
x = np.arange(len(impact.columns))
width = 0.34
for offset, (scenario, color, label) in enumerate([
    ("remove_low", TUM_BLUE, "Remove low-criticality pair"),
    ("remove_high", TUM_RED, "Remove high-criticality pair"),
]):
    values = impact.loc[scenario].to_numpy()
    bars = ax.bar(x + (offset - 0.5) * width, values, width, color=color, label=label)
    ax.bar_label(bars, fmt="%+.1f%%", fontsize=8, padding=2)
ax.axhline(0, color=TUM_GRAY_DARK, lw=0.9)
ax.set(
    xticks=x,
    xticklabels=impact.columns,
    ylabel="Change from baseline (%)",
    title="Operational impact of removing two employees",
)
ax.legend(fontsize=8)
fig.tight_layout()
save_figure(fig, "04_staffing_impact")
plt.show()

## 4. Reproducibility checks and interpretation guardrails

- The runner requires `lifecycle_mode` as a keyword, so future notebooks cannot silently fall back to legacy elapsed durations.
- Every run asserts an active `work_item_id` lifecycle log and checks its recorded configuration.
- CRN pairs the stochastic inputs by seed, making within-seed policy and staffing differences more informative than independent runs.
- Occupation is active busy time divided by calendar availability. Staff with zero availability in the horizon are reported and excluded from the denominator; staff who are available but idle count as zero occupation.
- Cycle time conditions on natural completion. Completion counts, completion shares, and milestone reach shares must therefore accompany it.
- The criticality ranking is only a screening device. The leave-two-out experiment estimates short-horizon operational effects; it does not include legal, ethical, financial, or employee-welfare considerations and should not be read as a real dismissal recommendation.

In [ ]:
assert (
    policy_run_metrics["resources_evaluated"]
    + policy_run_metrics["zero_availability_resources"]
).eq(len(human_resources)).all()
assert policy_run_metrics["completed_cases"].gt(0).all()
assert staffing_run_metrics["completed_cases"].gt(0).all()
assert sole_candidate_contexts.sum() == 0, "Update the report: the current model now contains a single-candidate context."

print("All provenance and result sanity checks passed.")
print("Results written to", RESULT_DIR)
print("Figures written to", ROOT / "visualization")